# Food inflation in Turkey

Can exchange rates and global food prices help predict monthly food inflation in Turkey?

- Monthly Turkish food CPI and four food categories: [TÜİK](https://veriportali.tuik.gov.tr/tr/press/58289), via [TCMB EVDS](https://evds3.tcmb.gov.tr/).
- Daily USD/TRY buying rates: [TCMB EVDS](https://evds3.tcmb.gov.tr/).
- Monthly global food-price indices: [FAO](https://www.fao.org/worldfoodsituation/foodpricesindex/en/).

## Setup

Run from the project folder. Install dependencies with `python -m pip install -r requirements.txt`.

In [1]:
import io
import os
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

raw_dir = Path("data/raw")

## Download or reuse saved data

Existing CSVs in `data/raw/` are reused. Delete a CSV to download that source again.

### Global food prices

In [2]:
raw_dir.mkdir(parents=True, exist_ok=True)
fao_path = raw_dir / "fao_food_prices.csv"

if fao_path.is_file():
    print(f"Using cached {fao_path.name}")
else:
    fao_url = (
        "https://www.fao.org/media/docs/worldfoodsituationlibraries/wfs-library/food_price_indices_data.csv?download=true&sfvrsn=523ebd2a_84"
    )
    response = requests.get(fao_url, timeout=90)
    response.raise_for_status()
    fao = pd.read_csv(io.BytesIO(response.content), skiprows=2, usecols=range(7)).dropna(how="all")
    fao.to_csv(fao_path, index=False)
    print(f"Saved fao_food_prices.csv: {len(fao)} months")

Using cached fao_food_prices.csv


### EVDS API key

A key is needed only when Turkish data files are missing. Get one from [EVDS](https://evds3.tcmb.gov.tr/) and save it as `EVDS_API_KEY` in `.env`, or enter it at the hidden prompt. See the [API guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16).

In [3]:
load_dotenv(".env")
api_key = os.getenv("EVDS_API_KEY", "").strip()

raw_dir.mkdir(parents=True, exist_ok=True)
end_date = date.today().replace(day=1) - timedelta(days=1)

### Turkish CPI and USD/TRY

Download January 2005 through the last completed month, in yearly requests. [EVDS dates](https://www.tcmb.gov.tr/wps/wcm/connect/bab69efb-d66c-45b5-91c6-1533886acd6e/GenelAg-SSS.pdf?CACHE=NONE&CONTENTCACHE=NONE&MOD=AJPERES&attachment=true) refer to rates determined on the previous business day.

In [4]:
series = {
    "turkish_food_cpi": [
        "TP.TUKFIY2025.01",
        "TP.TUKFIY2025.0111",
        "TP.TUKFIY2025.0112",
        "TP.TUKFIY2025.0114",
        "TP.TUKFIY2025.0115",
    ],
    "usd_try_daily": ["TP.DK.USD.A.YTL"],
}
for dataset, codes in series.items():
    csv_path = raw_dir / f"{dataset}.csv"
    if csv_path.is_file():
        print(f"Using cached {csv_path.name}")
        continue

    if not api_key:
        api_key = getpass("EVDS API key: ").strip()
        if not api_key:
            raise ValueError("Enter your EVDS API key in .env or the hidden prompt.")

    yearly_tables = []
    for year in range(2005, end_date.year + 1):
        year_end = min(date(year, 12, 31), end_date)
        url = (
            f"https://evds3.tcmb.gov.tr/igmevdsms-dis/series={'-'.join(codes)}&startDate=01-01-{year}&endDate={year_end:%d-%m-%Y}&type=csv"
        )
        response = requests.get(url, headers={"key": api_key}, timeout=90)
        response.raise_for_status()
        yearly_tables.append(pd.read_csv(io.BytesIO(response.content)))

    data = pd.concat(yearly_tables, ignore_index=True)
    data.to_csv(csv_path, index=False)
    print(f"Saved {dataset}.csv: {len(data)} observations")

Using cached turkish_food_cpi.csv
Using cached usd_try_daily.csv


## Load saved files

### Turkish food CPI

In [5]:
cpi = pd.read_csv("data/raw/turkish_food_cpi.csv").rename(columns={
    "Tarih": "date",
    "TP_TUKFIY2025_01": "food_cpi",
    "TP_TUKFIY2025_0111": "cereals_cpi",
    "TP_TUKFIY2025_0112": "meat_cpi",
    "TP_TUKFIY2025_0114": "dairy_eggs_cpi",
    "TP_TUKFIY2025_0115": "oils_fats_cpi",
    "UNIXTIME": "unix_time",
})

### Daily USD/TRY exchange rates

In [6]:
usd_try = pd.read_csv("data/raw/usd_try_daily.csv").rename(columns={
    "Tarih": "date",
    "TP_DK_USD_A_YTL": "usd_try",
    "UNIXTIME": "unix_time",
})

### Global food prices

In [7]:
fao = pd.read_csv("data/raw/fao_food_prices.csv").rename(columns={
    "Date": "date",
    "Food Price Index": "food_price_index",
    "Meat": "meat",
    "Dairy": "dairy",
    "Cereals": "cereals",
    "Oils": "vegetable_oils",
    "Sugar": "sugar",
})

### Source file sizes

Column counts include dates and metadata.

In [8]:
source_sizes = pd.DataFrame({
    "Rows": [len(cpi), len(usd_try), len(fao)],
    "Columns": [cpi.shape[1], usd_try.shape[1], fao.shape[1]],
    "One row": ["One month", "One calendar day", "One month"],
}, index=["Turkish CPI", "USD/TRY", "FAO"])
source_sizes

,Rows,Columns,One row
Turkish CPI,261,7,One month
USD/TRY,7943,3,One calendar day
FAO,441,7,One month


## Proposal data check

### Dates and numeric columns

In [9]:
cpi["date"] = pd.to_datetime(cpi["date"], format="%Y-%m", errors="raise")
usd_try["date"] = pd.to_datetime(usd_try["date"], format="%d-%m-%Y", errors="raise")
fao["date"] = pd.to_datetime(fao["date"], format="%Y-%m", errors="raise")

cpi = cpi.set_index("date").drop(columns="unix_time").apply(pd.to_numeric, errors="raise")
usd_try = usd_try.set_index("date").drop(columns="unix_time").apply(pd.to_numeric, errors="raise")
fao = fao.set_index("date").apply(pd.to_numeric, errors="raise")

### Date coverage

In [10]:
source_coverage = pd.DataFrame({
    "First date": [cpi.index.min().date(), usd_try.index.min().date(), fao.index.min().date()],
    "Last date": [cpi.index.max().date(), usd_try.index.max().date(), fao.index.max().date()],
    "Numeric series": [cpi.shape[1], usd_try.shape[1], fao.shape[1]],
}, index=["Turkish CPI", "USD/TRY", "FAO"])
source_coverage

,First date,Last date,Numeric series
Turkish CPI,2005-01-01,2026-09-01,5
USD/TRY,2005-01-01,2026-09-30,1
FAO,1990-01-01,2026-09-01,6


### Date checks

In [11]:
date_checks = pd.DataFrame({
    "All dates present": [cpi.index.notna().all(), usd_try.index.notna().all(), fao.index.notna().all()],
    "Unique dates": [cpi.index.is_unique, usd_try.index.is_unique, fao.index.is_unique],
    "In date order": [cpi.index.is_monotonic_increasing, usd_try.index.is_monotonic_increasing, fao.index.is_monotonic_increasing],
    "Missing calendar rows": [
        len(pd.date_range(cpi.index.min(), cpi.index.max(), freq="MS").difference(cpi.index)),
        len(pd.date_range(usd_try.index.min(), usd_try.index.max(), freq="D").difference(usd_try.index)),
        len(pd.date_range(fao.index.min(), fao.index.max(), freq="MS").difference(fao.index)),
    ],
}, index=["Turkish CPI", "USD/TRY", "FAO"])
assert date_checks[["All dates present", "Unique dates", "In date order"]].all().all()
date_checks

,All dates present,Unique dates,In date order,Missing calendar rows
Turkish CPI,True,True,True,0
USD/TRY,True,True,True,0
FAO,True,True,True,0


### Missing numeric values

In [12]:
source_missing = pd.concat({
    "Turkish CPI": cpi.isna().sum(),
    "USD/TRY": usd_try.isna().sum(),
    "FAO": fao.isna().sum(),
}, names=["Source", "Column"]).to_frame("Missing values")
source_missing

Missing values
Source      Column                          
Turkish CPI food_cpi                       0
            cereals_cpi                    0
            meat_cpi                       0
            dairy_eggs_cpi                 0
            oils_fats_cpi                  0
USD/TRY     usd_try                     2470
FAO         food_price_index               0
            meat                           0
            dairy                          0
            cereals                        0
            vegetable_oils                 0
            sugar                          0

### Missing daily exchange rates

In [13]:
missing_rate = usd_try["usd_try"].isna()
weekend = usd_try.index.dayofweek >= 5

exchange_rate_counts = pd.Series({
    "Published rates": int((~missing_rate).sum()),
    "Missing rates": int(missing_rate.sum()),
    "Missing on weekends": int((missing_rate & weekend).sum()),
    "Missing on weekdays": int((missing_rate & ~weekend).sum()),
}, name="Daily observations").to_frame()
exchange_rate_counts

,Daily observations
Published rates,5473
Missing rates,2470
Missing on weekends,2270
Missing on weekdays,200


## Monthly table

### Average daily exchange rates

Monthly means use available rates without filling daily gaps.

In [14]:
usd_try_monthly = usd_try.resample("MS").mean()
usd_try_monthly.head()

,usd_try
date,
2005-01-01,1.350042
2005-02-01,1.310155
2005-03-01,1.304957
2005-04-01,1.353514
2005-05-01,1.365048


### Join sources by month

Keep every month in the CPI date range.

In [15]:
monthly = cpi.asfreq("MS").join(usd_try_monthly, how="left")
monthly = monthly.join(fao.add_prefix("fao_"), how="left")
monthly.head()

,food_cpi,cereals_cpi,meat_cpi,dairy_eggs_cpi,oils_fats_cpi,usd_try,fao_food_price_index,fao_meat,fao_dairy,fao_cereals,fao_vegetable_oils,fao_sugar
date,,,,,,,,,,,,
2005-01-01,2.50,2.48,2.01,3.07,2.88,1.350042,65.8,69.9,76.2,60.5,63.15,54.0
2005-02-01,2.52,2.49,2.04,3.08,2.89,1.310155,66.2,71.1,76.7,60.3,62.27,56.3
2005-03-01,2.52,2.49,2.06,3.07,2.89,1.304957,67.5,71.7,77.0,61.5,66.74,55.0
2005-04-01,2.50,2.49,2.15,3.03,2.87,1.353514,66.0,70.4,76.2,59.5,65.07,53.2
2005-05-01,2.50,2.49,2.16,2.96,2.86,1.365048,66.5,72.6,76.5,59.4,64.23,53.1


### Monthly summary

The table contains index levels and exchange rates.

In [16]:
monthly_summary = pd.Series({
    "Monthly rows": len(monthly),
    "Numeric columns": monthly.shape[1],
    "First month": monthly.index.min().strftime("%B %Y"),
    "Last month": monthly.index.max().strftime("%B %Y"),
    "Missing numeric values": int(monthly.isna().sum().sum()),
    "Months missing any value": int(monthly.isna().any(axis=1).sum()),
    "Months without an exchange rate": int(monthly["usd_try"].isna().sum()),
}, name="Monthly coverage").to_frame()
monthly_summary

,Monthly coverage
Monthly rows,261
Numeric columns,12
First month,January 2005
Last month,September 2026
Missing numeric values,0
Months missing any value,0
Months without an exchange rate,0


In [17]:
monthly.isna().sum().to_frame("Missing monthly values")

,Missing monthly values
food_cpi,0
cereals_cpi,0
meat_cpi,0
dairy_eggs_cpi,0
oils_fats_cpi,0
usd_try,0
fao_food_price_index,0
fao_meat,0
fao_dairy,0
fao_cereals,0
